# SinoNom OCR — Phase 3: Alignment, Validation & Export

**HCMUS NaturalLanguageProcessing — Midterm Project**  
**An Nam Nhất Thống Chí (HVH_004)**

---

This notebook handles **Phase 3** of the pipeline: loading layout columns, aligning the OCR character output against transliterated Quoc Ngu reference text (using S1 ∩ S2 Levenshtein spelling correction), generating hierarchical sentence IDs, and exporting to structured XML and Excel sheets.


## 0 · Environment Setup


In [ ]:
# ─── Detect environment ────────────────────────────────────────────────────
import os
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")
print(f"Python {sys.version}")

In [ ]:
# ─── Google Drive Mount & Project Path Discovery (Colab only) ────────────────
if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    repo_name = "SinoNomViet_Transliteration_OCR"
    repo_url = f"https://github.com/khang3004/{repo_name}.git"
    candidate_files = [
        "src/sinonom_ocr/data_scraper.py",
        "src/sinonom_ocr/spatial_layout_engine.py",
        "src/sinonom_ocr/alignment_validator.py",
    ]

    def has_project_files(path: Path) -> bool:
        return all((path / f).exists() for f in candidate_files)

    found_root = None

    # 1. Check if files are already in current directory or a subfolder of /content
    cwd = Path(os.getcwd())
    if has_project_files(cwd):
        found_root = cwd
    else:
        for p in Path("/content").glob("**/src/sinonom_ocr/data_scraper.py"):
            if has_project_files(p.parent.parent.parent):
                found_root = p.parent.parent.parent
                break

    # 2. If not found locally, try to find them on Google Drive
    if not found_root:
        drive_mounted = os.path.exists("/content/drive/MyDrive")
        if not drive_mounted:
            print("❓ Google Drive is not mounted. Mounting now...")
            try:
                from google.colab import drive

                drive.mount("/content/drive")
                drive_mounted = True
            except Exception as e:
                print(f"⚠️ Google Drive mount failed or skipped: {e}")

        if drive_mounted:
            drive_paths = [
                Path("/content/drive/MyDrive/SinoNomOCR/HVH_004"),
                Path(f"/content/drive/MyDrive/{repo_name}"),
                Path("/content/drive/MyDrive/SinoNomVietnamese_OCR_Project"),
                Path("/content/drive/MyDrive/SinoNomViet_Transliteration_OCR"),
            ]
            for dp in drive_paths:
                if dp.exists() and has_project_files(dp):
                    found_root = dp
                    print(f"🎯 Found project files in Drive: {found_root}")
                    break

            if not found_root:
                print("🔍 Searching Google Drive for project files...")
                for p in Path("/content/drive/MyDrive").glob("**/src/sinonom_ocr/data_scraper.py"):
                    if has_project_files(p.parent.parent.parent):
                        found_root = p.parent.parent.parent
                        print(f"🎯 Found project files in Drive subfolder: {found_root}")
                        break

    # 3. If still not found, clone the repository from GitHub
    if not found_root:
        print("📁 Project files not found. Cloning repository from GitHub...")
        try:
            subprocess.run(["git", "clone", repo_url, f"/content/{repo_name}"], check=True)
            found_root = Path(f"/content/{repo_name}")
            print("✅ Repository cloned successfully.")
        except Exception as e:
            print(f"❌ Failed to clone repository: {e}")

    if found_root:
        PROJECT_ROOT = str(found_root)
        os.chdir(PROJECT_ROOT)
        print(f"🎯 Project root set to: {PROJECT_ROOT}")
        print(f"🔄 Changed working directory to: {os.getcwd()}")
    else:
        PROJECT_ROOT = "/content"
        print("❌ ERROR: Could not locate project root.")
else:
    # Local Jupyter running in notebooks/ directory
    PROJECT_ROOT = os.path.abspath("..")
    os.chdir(PROJECT_ROOT)
    print(f"🎯 Local project root: {PROJECT_ROOT}")
    print(f"🔄 Changed working directory to: {os.getcwd()}")

In [ ]:
# ─── Install package & dependencies ────────────────────────────────────────
# Installs the package in editable mode, which resolves all dependencies from pyproject.toml
if IN_COLAB:
    # Install the package in editable mode to register sinonom_ocr package
    %pip install -q -e .
    print("✅ Package and dependencies installed on Colab.")
else:
    print("✅ Local environment is managed by uv (run 'make install' if needed).")

In [ ]:
# ─── Path configuration ────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path(PROJECT_ROOT)

# Add src/ directory to sys.path so we can import our sinonom_ocr package
SRC_PATH = ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Detect Google Drive paths (Local Mac or Colab)
LOCAL_DRIVE_PATH = Path("/Users/KhangDS/Library/CloudStorage/GoogleDrive-gausseuler159357@gmail.com/My Drive/SinoNomOCR/HVH_004")
COLAB_DRIVE_PATH = Path("/content/drive/MyDrive/SinoNomOCR/HVH_004")

# Determine active directories
if IN_COLAB and COLAB_DRIVE_PATH.exists():
    print("✨ Using Google Drive directories on Google Colab.")
    DATA_DIR = COLAB_DRIVE_PATH / "data"
    OUTPUT_DIR = COLAB_DRIVE_PATH / "output"
elif LOCAL_DRIVE_PATH.exists():
    print("✨ Using Google Drive directories on local macOS.")
    DATA_DIR = LOCAL_DRIVE_PATH / "data"
    OUTPUT_DIR = LOCAL_DRIVE_PATH / "output"
else:
    print("✨ Using local repository directories.")
    DATA_DIR = ROOT / "data"
    OUTPUT_DIR = ROOT / "output"

# Output directories
RAW_IMAGES_DIR = DATA_DIR / "raw_images"
OUTPUT_XML_DIR = OUTPUT_DIR / "xml"
OUTPUT_XLSX_DIR = OUTPUT_DIR / "excel"
DICT_DIR = DATA_DIR / "dicts"

for d in [RAW_IMAGES_DIR, OUTPUT_XML_DIR, OUTPUT_XLSX_DIR, DICT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Copy reference dictionary files if missing from target DICT_DIR
repo_dicts_dir = ROOT / "data" / "dicts"
if repo_dicts_dir.exists() and DICT_DIR != repo_dicts_dir:
    import shutil
    for dic_file in repo_dicts_dir.glob("*.dic"):
        target_file = DICT_DIR / dic_file.name
        if not target_file.exists():
            shutil.copy(dic_file, target_file)
            print(f"📋 Copied reference dictionary {dic_file.name} to {target_file}")

print("Directory structure initialized.")

---

## 1 · Import Alignment & Validation Modules


In [ ]:
import json
import logging
import re
from datetime import datetime
import requests
import csv
import ast

# Automatically download hanviet.csv if missing
csv_path = DICT_DIR / "hanviet.csv"
if not csv_path.exists():
    print("📥 Downloading hanviet.csv dictionary database...")
    url = "https://raw.githubusercontent.com/ph0ngp/hanviet-pinyin-wordlist/master/hanviet.csv"
    resp = requests.get(url)
    resp.raise_for_status()
    csv_path.write_text(resp.text, encoding="utf-8")
    print("✅ hanviet.csv downloaded successfully.")

from sinonom_ocr.alignment_validator import (
    QUOCNGU_SINONOM_S2,
    SINONOM_SIMILAR_S1,
    AlignmentStatus,
    CharAlignmentResult,
    SinoNomAlignmentValidator,
)
from sinonom_ocr.spatial_layout_engine import BoundingBox, Column

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)-8s | %(name)s | %(message)s",
)

# Load validator with our augmented dictionary
validator = SinoNomAlignmentValidator(hanviet_path=str(csv_path))

# Build character-to-reading map from hanviet.csv for automated Hán-Việt transliteration
char_to_qn = {}
with open(csv_path, encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)  # Skip header
    for row in reader:
        if len(row) < 2:
            continue
        char = row[0].strip()
        raw_readings = row[1].strip()
        try:
            readings = ast.literal_eval(raw_readings)
        except:
            readings = [r.strip("'\" ") for r in raw_readings.strip("[]").split(",")]
        if char and readings:
            char_to_qn[char] = readings[0].strip().lower()

print("✅ Alignment & Validation modules loaded and initialized with HanViet CSV.")

---

## 2 · Sentence ID Generator

Prof. Dien's 14-character hierarchical ID format:

```
DSG_fff.ccc.ppp.ss
```

For _An Nam Nhất Thống Chí_ the prefix is `HVH_004`:

- `H` = History domain
- `V` = (sub-domain: Vietnamese)
- `H` = (genre: Hán, vertical, carved-printed)
- `_004` = file index 004 within that sub-domain


In [ ]:
# ─── Sentence ID schema ────────────────────────────────────────────────────


class SentenceIDGenerator:
    """Generate systematic 14-character hierarchical sentence IDs.

    Format: ``DSG_fff.ccc.ppp.ss``

    For An Nam Nhất Thống Chí (HVH_004):
        - D   = 'H' (History)
        - S   = 'V' (Vietnamese historical sub-domain)
        - G   = 'H' (Hán script, vertical, carved-print)
        - fff = '004' (file index 004)
        - ccc = chapter number (001–999)
        - ppp = page number within chapter (001–999)
        - ss  = sentence/box number within page (01–99)

    Full example: ``HVH_004.001.003.07``
    """

    CORPUS_PREFIX = "HVH_004"  # Fixed prefix for this work

    @classmethod
    def generate(
        cls,
        chapter: int,
        page: int,
        sentence: int,
    ) -> str:
        """Generate one sentence ID.

        Args:
            chapter:  Chapter number (1-based).
            page:     Page number within chapter (1-based).
            sentence: Sentence/box number within page (1-based).

        Returns:
            14-character ID string, e.g. ``HVH_004.001.003.07``

        Raises:
            ValueError: If any argument is out of valid range.
        """
        if not (1 <= chapter <= 999):
            raise ValueError(f"chapter must be 1–999, got {chapter}")
        if not (1 <= page <= 999):
            raise ValueError(f"page must be 1–999, got {page}")
        if not (1 <= sentence <= 99):
            raise ValueError(f"sentence must be 1–99, got {sentence}")

        sid = f"{cls.CORPUS_PREFIX}.{chapter:03d}.{page:03d}.{sentence:02d}"
        assert len(sid) == 18, f"ID length mismatch: {sid!r} has {len(sid)} chars"
        return sid

    @classmethod
    def parse(cls, sentence_id: str) -> dict[str, int]:
        """Parse a sentence ID back into its component fields.

        Args:
            sentence_id: A string like ``HVH_004.001.003.07``.

        Returns:
            Dictionary with keys: chapter, page, sentence.

        Raises:
            ValueError: If the ID format is invalid.
        """
        pattern = r"^([A-Z]{3}_\d{3})\.(\d{3})\.(\d{3})\.(\d{2})$"
        m = re.match(pattern, sentence_id)
        if not m:
            raise ValueError(f"Invalid sentence ID format: {sentence_id!r}")
        return {
            "prefix": m.group(1),
            "chapter": int(m.group(2)),
            "page": int(m.group(3)),
            "sentence": int(m.group(4)),
        }


# ─── Verification ─────────────────────────────────────────────────────────
gen = SentenceIDGenerator()

test_ids = [
    (1, 1, 1),
    (1, 3, 7),
    (12, 45, 23),
    (999, 999, 99),
]
print("Sentence ID generation examples:")
for ch, pg, sn in test_ids:
    sid = SentenceIDGenerator.generate(ch, pg, sn)
    parsed = SentenceIDGenerator.parse(sid)
    print(f"  ({ch:3d}, {pg:3d}, {sn:2d}) → {sid!r}  len={len(sid)}  ✓parse={parsed}")

---

## 3 · Load Spatial Layout Results from Phase 2

We load the spatial layout columns and boxes exported as JSON from Phase 2, and reconstruct the Python object structure needed for alignment validation.


In [ ]:
# ─── Deserialize layout from JSON ──────────────────────────────────────────

input_file = DATA_DIR / "ocr_layout_output.json"

with open(input_file, encoding="utf-8") as fh:
    serialized_corpus = json.load(fh)

corpus_layout = []
for ch_idx, ch_data in enumerate(serialized_corpus, start=1):
    ch_layout = []
    for pg_idx, pg_data in enumerate(ch_data, start=1):
        # Reconstruct BoundingBoxes for ordered list
        ordered_boxes = [
            BoundingBox(
                raw_polygon=b["raw_polygon"],
                text=b["text"],
                confidence=b["confidence"],
                column_id=b["column_id"],
                reading_idx=b["reading_idx"],
            )
            for b in pg_data["ordered_boxes"]
        ]

        # Reconstruct Columns
        columns = []
        for col_data in pg_data["columns"]:
            col_boxes = [
                BoundingBox(
                    raw_polygon=b["raw_polygon"],
                    text=b["text"],
                    confidence=b["confidence"],
                    column_id=b["column_id"],
                    reading_idx=b["reading_idx"],
                )
                for b in col_data["boxes"]
            ]
            col = Column(
                column_id=col_data["column_id"],
                boxes=col_boxes,
                x_center=col_data["x_center"],
                x_span=tuple(col_data["x_span"]),
            )
            columns.append(col)

        ch_layout.append((columns, ordered_boxes))
    corpus_layout.append(ch_layout)

print(f"✅ Successfully loaded and deserialized layout results for {len(corpus_layout)} chapters.")

---

## 5 · Mock Quoc Ngu (QN) Reference Data

In production, these come from a bilingual alignment file or TTS romanisation.


In [ ]:
# ─── Generate dynamic Quoc Ngu reference sequences from OCR characters ─────
# Mirrors the corpus structure: corpus_qn[chapter][page][column] = [qn_words]

CORPUS_QN: list[list[list[list[str]]]] = []
for ch_layout in corpus_layout:
    ch_qn = []
    for columns, ordered_boxes in ch_layout:
        pg_qn = []
        for col in columns:
            col_qn = []
            for box in col.boxes:
                for char in box.text:
                    qn_word = char_to_qn.get(char, char)
                    col_qn.append(qn_word)
            pg_qn.append(col_qn)
        ch_qn.append(pg_qn)
    CORPUS_QN.append(ch_qn)

print("✅ Dynamically generated Hán-Việt references matching OCR layout.")
print(f"Chapters: {len(CORPUS_QN)}")
for ci, ch in enumerate(CORPUS_QN, 1):
    for pi, pg in enumerate(ch, 1):
        print(f"  Ch{ci} Pg{pi}: {len(pg)} columns")

---

## 6 · Alignment Validation Pipeline


In [ ]:
# ─── Run full alignment validation ────────────────────────────────────────
from dataclasses import dataclass


@dataclass
class AlignedSentence:
    """One fully-processed sentence record, ready for XML/Excel export."""

    sentence_id: str
    chapter: int
    page: int
    column: int  # 0-based, right-to-left
    sn_chars: list[str]
    qn_words: list[str]
    char_results: list[CharAlignmentResult]
    edit_distance: int
    accuracy: float
    bbox_xmin: float
    bbox_ymin: float
    bbox_xmax: float
    bbox_ymax: float


validator = SinoNomAlignmentValidator()
all_sentences: list[AlignedSentence] = []

for ch_idx, (chapter_layout, chapter_qn) in enumerate(zip(corpus_layout, CORPUS_QN), start=1):
    for pg_idx, ((columns, ordered_boxes), page_qn) in enumerate(
        zip(chapter_layout, chapter_qn), start=1
    ):
        # Each column = one "sentence" (box group)
        for col_idx, (col, col_qn) in enumerate(zip(columns, page_qn), start=1):
            # Generate sentence ID
            sid = SentenceIDGenerator.generate(ch_idx, pg_idx, col_idx)

            # Extract character sequences
            sn_seq = [char for b in col.boxes for char in b.text]
            qn_seq = col_qn  # From reference data

            # Run alignment validation
            seq_result = validator.validate_sequence(sn_seq, qn_seq)

            # Compute combined bounding box for the column
            all_x = [b.x_min for b in col.boxes] + [b.x_max for b in col.boxes]
            all_y = [b.y_min for b in col.boxes] + [b.y_max for b in col.boxes]

            sentence = AlignedSentence(
                sentence_id=sid,
                chapter=ch_idx,
                page=pg_idx,
                column=col.column_id,
                sn_chars=sn_seq,
                qn_words=qn_seq,
                char_results=seq_result.char_results,
                edit_distance=seq_result.edit_distance,
                accuracy=seq_result.accuracy,
                bbox_xmin=min(all_x),
                bbox_ymin=min(all_y),
                bbox_xmax=max(all_x),
                bbox_ymax=max(all_y),
            )
            all_sentences.append(sentence)

print(f"✅ Processed {len(all_sentences)} sentences.")
print("\nSample results:")
for s in all_sentences[:3]:
    print(f"  [{s.sentence_id}] SN={s.sn_chars} QN={s.qn_words} acc={s.accuracy:.0%}")

In [ ]:
# ─── Alignment summary statistics ─────────────────────────────────────────
import statistics

status_counts = {s.value: 0 for s in AlignmentStatus}
for sentence in all_sentences:
    for cr in sentence.char_results:
        status_counts[cr.status.value] += 1

total_chars = sum(status_counts.values())
accuracies = [s.accuracy for s in all_sentences]

print("=" * 50)
print("Alignment Statistics")
print("=" * 50)
print(f"Total sentences  : {len(all_sentences)}")
print(f"Total characters : {total_chars}")
print()
for status, count in status_counts.items():
    pct = count / total_chars * 100 if total_chars else 0
    bar = "█" * int(pct / 2)
    print(f"  {status:6s}: {count:4d} ({pct:5.1f}%)  {bar}")
print()
print(f"Mean accuracy    : {statistics.mean(accuracies):.1%}")
print(f"Median accuracy  : {statistics.median(accuracies):.1%}")

---

## 7 · XML Export

Generates a standards-compliant XML file with strict metadata headers as
required by Prof. Dien's specification (Section A.6 of MidTerm_Requirement.pdf).


In [ ]:
# ─── XML export ────────────────────────────────────────────────────────────
from xml.dom import minidom
from xml.etree import ElementTree as ET


def build_xml(sentences: list[AlignedSentence]) -> ET.Element:
    """Build the complete XML document tree.

    XML schema follows Prof. Dien's specification:
    - Root: <corpus> with full metadata
    - Per chapter: <chapter id="...">
    - Per page:    <page id="...">
    - Per sentence: <sentence id="HVH_004.ccc.ppp.ss">
      - <han>    ... SinoNom characters
      - <quocngu>... Romanised Vietnamese
      - <alignment> per-character results

    Args:
        sentences: List of fully processed :class:`AlignedSentence` objects.

    Returns:
        Root XML ``Element``.
    """
    # ── Root corpus element with metadata ──────────────────────────────────
    root = ET.Element("corpus")
    root.set("xmlns:xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root.set("version", "1.0")

    # Metadata header (Prof. Dien requirement: tên tác phẩm, tác giả, ...)
    meta = ET.SubElement(root, "metadata")
    meta_fields = {
        "title": "An Nam Nhất Thống Chí",
        "title_han": "安南一統志",
        "author": "Lê Quang Định",
        "dynasty": "Nguyễn",
        "period": "1806",
        "language": "Classical Chinese / Hán",
        "script": "Han (Chữ Hán)",
        "source": "Nom Foundation Digital Library",
        "source_url": "https://lib.nomfoundation.org/collection/1/volume/664/",
        "corpus_id": "HVH_004",
        "domain": "H — History (Lịch sử)",
        "sub_domain": "VH — Vietnamese Historical Records",
        "processing_date": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        "pipeline_version": "1.0.0",
        "generator": "HCMUS NLP Pipeline — SinoNomVietnamese_OCR_Project",
    }
    for key, val in meta_fields.items():
        elem = ET.SubElement(meta, key)
        elem.text = val

    # ── Organise sentences by chapter → page ──────────────────────────────
    from itertools import groupby

    sorted_sents = sorted(sentences, key=lambda s: (s.chapter, s.page, s.column))

    body = ET.SubElement(root, "body")

    for ch_num, ch_group in groupby(sorted_sents, key=lambda s: s.chapter):
        ch_elem = ET.SubElement(body, "chapter")
        ch_elem.set("id", f"{ch_num:03d}")
        ch_elem.set("number", str(ch_num))

        for pg_num, pg_group in groupby(ch_group, key=lambda s: s.page):
            pg_elem = ET.SubElement(ch_elem, "page")
            pg_elem.set("id", f"{pg_num:03d}")
            pg_elem.set("number", str(pg_num))

            for sentence in pg_group:
                stc = ET.SubElement(pg_elem, "sentence")
                stc.set("id", sentence.sentence_id)
                stc.set("column", str(sentence.column))
                stc.set("accuracy", f"{sentence.accuracy:.4f}")
                stc.set("edit_distance", str(sentence.edit_distance))

                # Bounding box
                bbox_elem = ET.SubElement(stc, "bbox")
                bbox_elem.set("x_min", f"{sentence.bbox_xmin:.1f}")
                bbox_elem.set("y_min", f"{sentence.bbox_ymin:.1f}")
                bbox_elem.set("x_max", f"{sentence.bbox_xmax:.1f}")
                bbox_elem.set("y_max", f"{sentence.bbox_ymax:.1f}")

                # Han text
                han_elem = ET.SubElement(stc, "han")
                han_elem.text = "".join(sentence.sn_chars)

                # Quoc Ngu text
                qn_elem = ET.SubElement(stc, "quocngu")
                qn_elem.text = " ".join(sentence.qn_words)

                # Character-level alignment
                align_elem = ET.SubElement(stc, "alignment")
                for cr in sentence.char_results:
                    pair = ET.SubElement(align_elem, "pair")
                    pair.set("sn", cr.sn)
                    pair.set("qn", cr.qn)
                    pair.set("status", cr.status.value)
                    if cr.corrected_char:
                        pair.set("corrected", cr.corrected_char)

    return root


def prettify_xml(element: ET.Element) -> str:
    """Return a pretty-printed XML string with declaration header."""
    rough_string = ET.tostring(element, encoding="unicode")
    reparsed = minidom.parseString('<?xml version="1.0" encoding="UTF-8"?>\n' + rough_string)
    return reparsed.toprettyxml(indent="  ", encoding=None)


# Build and save
xml_root = build_xml(all_sentences)
xml_str = prettify_xml(xml_root)

xml_output_path = OUTPUT_XML_DIR / "HVH_004_corpus.xml"
xml_output_path.write_text(xml_str, encoding="utf-8")

print(f"✅ XML written to: {xml_output_path}")
print(f"   File size: {xml_output_path.stat().st_size:,} bytes")
print()
# Preview first 40 lines
lines = xml_str.splitlines()
print("\n".join(lines[:40]))
if len(lines) > 40:
    print(f"  ... ({len(lines) - 40} more lines)")

---

## 8 · Excel Export

Generates the alignment map Excel file as required by Prof. Dien:

- **Sheet 1**: Sentence-level summary (ID, Han text, QN text, accuracy)
- **Sheet 2**: Character-level alignment (per char: sn, qn, status, corrected)


In [ ]:
# ─── Excel export ──────────────────────────────────────────────────────────
import openpyxl
from openpyxl.styles import (
    Alignment as XlAlign,
)
from openpyxl.styles import (
    Border,
    Font,
    PatternFill,
    Side,
)
from openpyxl.utils import get_column_letter

# ── Colour palette (matching Prof. Dien's BLACK/GREEN/RED convention) ──────
FILL_BLACK = PatternFill(fill_type="solid", fgColor="333333")  # Header
FILL_GREEN = PatternFill(fill_type="solid", fgColor="C6EFCE")  # Corrected
FILL_RED = PatternFill(fill_type="solid", fgColor="FFC7CE")  # Failure
FILL_WHITE = PatternFill(fill_type="solid", fgColor="FFFFFF")  # Correct
FILL_HEADER = PatternFill(fill_type="solid", fgColor="1F4E79")  # Header
FILL_ALT = PatternFill(fill_type="solid", fgColor="D9E1F2")  # Alt row

FONT_HEADER = Font(bold=True, color="FFFFFF", size=11, name="Calibri")
FONT_HAN = Font(name="SimSun", size=12)  # CJK-compatible font
FONT_NORMAL = Font(name="Calibri", size=10)

BORDER_THIN = Border(
    left=Side(style="thin"),
    right=Side(style="thin"),
    top=Side(style="thin"),
    bottom=Side(style="thin"),
)


def style_header_row(ws, row: int, col_count: int) -> None:
    """Apply header styling to an entire row."""
    for col in range(1, col_count + 1):
        cell = ws.cell(row=row, column=col)
        cell.font = FONT_HEADER
        cell.fill = FILL_HEADER
        cell.alignment = XlAlign(horizontal="center", vertical="center", wrap_text=True)
        cell.border = BORDER_THIN


def auto_width(ws) -> None:
    """Auto-fit column widths (approximate)."""
    for col_cells in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col_cells[0].column)
        for cell in col_cells:
            val = str(cell.value or "")
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 40)


# ── Build workbook ─────────────────────────────────────────────────────────
wb = openpyxl.Workbook()

# ── Sheet 1: Sentence Summary ──────────────────────────────────────────────
ws1 = wb.active
ws1.title = "Sentence Summary"

# Title row
ws1.merge_cells("A1:J1")
title_cell = ws1["A1"]
title_cell.value = "HVH_004 — An Nam Nhất Thống Chí | OCR Alignment Dataset"
title_cell.font = Font(bold=True, size=14, color="1F4E79", name="Calibri")
title_cell.alignment = XlAlign(horizontal="center", vertical="center")
ws1.row_dimensions[1].height = 28

# Metadata row
ws1["A2"] = f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
ws1["A2"].font = Font(italic=True, size=9, color="666666")
ws1.merge_cells("A2:J2")

# Header row
S1_HEADERS = [
    "Sentence ID",
    "Chapter",
    "Page",
    "Column",
    "Hán Tự (SN)",
    "Quốc Ngữ (QN)",
    "Edit Distance",
    "Accuracy",
    "Status",
    "Notes",
]
for col_idx, header in enumerate(S1_HEADERS, 1):
    ws1.cell(row=3, column=col_idx, value=header)
style_header_row(ws1, 3, len(S1_HEADERS))
ws1.row_dimensions[3].height = 18
ws1.freeze_panes = "A4"

# Data rows
for row_idx, s in enumerate(all_sentences, start=4):
    fill = FILL_ALT if row_idx % 2 == 0 else FILL_WHITE
    overall_status = (
        "✅ Good" if s.accuracy >= 0.8 else "⚠️ Partial" if s.accuracy >= 0.5 else "❌ Poor"
    )
    row_data = [
        s.sentence_id,
        s.chapter,
        s.page,
        s.column,
        "".join(s.sn_chars),
        " ".join(s.qn_words),
        s.edit_distance,
        f"{s.accuracy:.1%}",
        overall_status,
        "",
    ]
    for col_idx, val in enumerate(row_data, 1):
        cell = ws1.cell(row=row_idx, column=col_idx, value=val)
        cell.border = BORDER_THIN
        cell.fill = fill
        if col_idx == 5:  # Han characters
            cell.font = FONT_HAN
            cell.alignment = XlAlign(horizontal="center")
        else:
            cell.font = FONT_NORMAL
            cell.alignment = XlAlign(horizontal="left", vertical="center")

auto_width(ws1)

# ── Sheet 2: Character Alignment Detail ───────────────────────────────────
ws2 = wb.create_sheet("Character Alignment")

ws2.merge_cells("A1:G1")
ws2["A1"] = "Character-Level Alignment Detail — S1∩S2 Algorithm"
ws2["A1"].font = Font(bold=True, size=13, color="1F4E79")
ws2["A1"].alignment = XlAlign(horizontal="center")

S2_HEADERS = [
    "Sentence ID",
    "Position",
    "SinoNom (sn)",
    "Quốc Ngữ (qn)",
    "Status",
    "Corrected Char",
    "Confidence",
]
for col_idx, header in enumerate(S2_HEADERS, 1):
    ws2.cell(row=2, column=col_idx, value=header)
style_header_row(ws2, 2, len(S2_HEADERS))
ws2.freeze_panes = "A3"

current_row = 3
for s in all_sentences:
    for pos, cr in enumerate(s.char_results, 1):
        # Status-based cell fill
        if cr.status == AlignmentStatus.BLACK:
            status_fill = FILL_WHITE
            status_label = "⚫ BLACK (Correct)"
        elif cr.status == AlignmentStatus.GREEN:
            status_fill = FILL_GREEN
            status_label = "🟢 GREEN (Corrected)"
        else:
            status_fill = FILL_RED
            status_label = "🔴 RED (Failure)"

        row_data = [
            s.sentence_id,
            pos,
            cr.sn,
            cr.qn,
            status_label,
            cr.corrected_char or "",
            "",  # Confidence placeholder
        ]
        for col_idx, val in enumerate(row_data, 1):
            cell = ws2.cell(row=current_row, column=col_idx, value=val)
            cell.border = BORDER_THIN
            cell.font = FONT_HAN if col_idx in (3, 6) else FONT_NORMAL
            cell.alignment = XlAlign(
                horizontal="center" if col_idx in (2, 3, 4, 6) else "left",
                vertical="center",
            )
            if col_idx == 5:  # Status column — apply colour
                cell.fill = status_fill
        current_row += 1

auto_width(ws2)

# ── Sheet 3: Dictionary Reference ─────────────────────────────────────────
ws3 = wb.create_sheet("Dictionary Reference")
ws3["A1"] = "S1: SinoNom Similar Characters (partial — built-in)"
ws3["A1"].font = Font(bold=True, size=12)
ws3.cell(row=2, column=1, value="SinoNom Char (sn)")
ws3.cell(row=2, column=2, value="Similar Chars (S1 ordered list)")
style_header_row(ws3, 2, 2)
for r, (char, similars) in enumerate(SINONOM_SIMILAR_S1.items(), start=3):
    ws3.cell(row=r, column=1, value=char).font = FONT_HAN
    ws3.cell(row=r, column=2, value=" ".join(similars)).font = FONT_HAN
    ws3.cell(row=r, column=1).border = BORDER_THIN
    ws3.cell(row=r, column=2).border = BORDER_THIN

# S2 dictionary table
s2_start = len(SINONOM_SIMILAR_S1) + 5
ws3.cell(row=s2_start, column=1, value="S2: QuocNgu → SinoNom Translations (partial)")
ws3.cell(row=s2_start, column=1).font = Font(bold=True, size=12)
ws3.cell(row=s2_start + 1, column=1, value="Quoc Ngu Word (qn)")
ws3.cell(row=s2_start + 1, column=2, value="SinoNom Candidates (S2)")
style_header_row(ws3, s2_start + 1, 2)
for r, (word, chars) in enumerate(QUOCNGU_SINONOM_S2.items(), start=s2_start + 2):
    ws3.cell(row=r, column=1, value=word).font = FONT_NORMAL
    ws3.cell(row=r, column=2, value=" ".join(sorted(chars))).font = FONT_HAN
    ws3.cell(row=r, column=1).border = BORDER_THIN
    ws3.cell(row=r, column=2).border = BORDER_THIN

auto_width(ws3)

# ── Save workbook ──────────────────────────────────────────────────────────
xlsx_path = OUTPUT_XLSX_DIR / "HVH_004_alignment_map.xlsx"
wb.save(xlsx_path)

print(f"✅ Excel written to: {xlsx_path}")
print(f"   File size: {xlsx_path.stat().st_size:,} bytes")
print(f"   Sheets: {[s.title for s in wb.worksheets]}")

---

## 9 · Validation & Sanity Checks


In [ ]:
# ─── Validate XML output ───────────────────────────────────────────────────
from xml.etree import ElementTree as ET

tree = ET.parse(xml_output_path)
xml_root_check = tree.getroot()

sentences_in_xml = xml_root_check.findall(".//sentence")

print("XML Validation")
print("=" * 50)
print(f"  Root tag      : {xml_root_check.tag}")
print(f"  Total sentences: {len(sentences_in_xml)}")
print(f"  Expected      : {len(all_sentences)}")
assert len(sentences_in_xml) == len(all_sentences), "Sentence count mismatch!"

# Validate all IDs follow the 18-char format HVH_004.ccc.ppp.ss
id_pattern = re.compile(r"^[A-Z]{3}_\d{3}\.\d{3}\.\d{3}\.\d{2}$")
for s_elem in sentences_in_xml:
    sid = s_elem.get("id", "")
    assert id_pattern.match(sid), f"Invalid ID format: {sid!r}"

print(f"  All {len(sentences_in_xml)} IDs validated ✓")
print()

# Sample XML content
print("Sample sentence XML:")
first_sent = sentences_in_xml[0]
ET.indent(first_sent, space="  ")
print(ET.tostring(first_sent, encoding="unicode"))

In [ ]:
# ─── Validate Excel output ─────────────────────────────────────────────────
wb_check = openpyxl.load_workbook(xlsx_path)
ws1_check = wb_check["Sentence Summary"]

data_rows = ws1_check.max_row - 3  # Subtract title + meta + header rows
print("Excel Validation")
print("=" * 50)
print(f"  Sheets        : {wb_check.sheetnames}")
print(f"  Sheet 1 rows  : {data_rows} data rows (excl. 3 header rows)")
print(f"  Expected      : {len(all_sentences)}")
assert data_rows == len(all_sentences), "Excel row count mismatch!"
print("  ✅ Row count validated")

print()
print("Pipeline complete! Output files:")
print(f"  📄 XML  : {xml_output_path}")
print(f"  📊 XLSX : {xlsx_path}")

---

## 10 · Pipeline Summary Report


In [ ]:
# ─── Final summary ─────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════╗")
print("║      HVH_004 — End-to-End Pipeline Summary          ║")
print("╠══════════════════════════════════════════════════════╣")
print("║  Corpus        : An Nam Nhất Thống Chí (HVH_004)   ║")
print(f"║  Chapters      : {len(corpus_layout):<36} ║")
total_pages = sum(len(ch) for ch in corpus_layout)
print(f"║  Pages         : {total_pages:<36} ║")
print(f"║  Sentences     : {len(all_sentences):<36} ║")
print(f"║  Total chars   : {total_chars:<36} ║")
print(
    f"║  BLACK (OK)    : {status_counts['BLACK']:>4} ({status_counts['BLACK'] / max(total_chars, 1) * 100:5.1f}%)                    ║"
)
print(
    f"║  GREEN (fixed) : {status_counts['GREEN']:>4} ({status_counts['GREEN'] / max(total_chars, 1) * 100:5.1f}%)                    ║"
)
print(
    f"║  RED (error)   : {status_counts['RED']:>4} ({status_counts['RED'] / max(total_chars, 1) * 100:5.1f}%)                    ║"
)
mean_acc = statistics.mean(s.accuracy for s in all_sentences)
mean_acc_str = f"{mean_acc:.1%}"
print(f"║  Mean accuracy : {mean_acc_str:<36} ║")
print("╠══════════════════════════════════════════════════════╣")
print("║  XML output    : HVH_004_corpus.xml                 ║")
print("║  XLSX output   : HVH_004_alignment_map.xlsx         ║")
print("╚══════════════════════════════════════════════════════╝")